## MMTHE01 - Masters Thesis

### A. Thesis - Data Cleansing
* Dealing with missing data

#### Importing the libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os

from sklearn.preprocessing import LabelEncoder

In [ ]:
# Check your current working directory
cwd = os.getcwd()

# Define your relative path
relative_path = r"Analysis"  # adjust this relative to cwd

# Build the full path
full_path = os.path.join(cwd, relative_path)

# Check if it exists before changing
if os.path.exists(full_path):
    os.chdir(full_path)
    print("Changed directory to:", full_path)
else:
    print("Folder does not exist:", full_path)

#### Importing the dataset

In [3]:
dataset = pd.read_csv('final_lending_club.csv', low_memory=False)

In [4]:
dataset.shape

(226067, 147)

### Step 1: Preliminary feature elimination

#### 1.1 Eliminating features which are only available after loan origination

In [5]:
features_to_eliminate_lo = ['pymnt_plan', 'out_prncp', 'out_prncp_inv',  'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 
                         'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
                         'last_credit_pull_d', 'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 
                         'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'hardship_length', 'hardship_dpd',
                         'hardship_loan_status', 'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount', 'installment', 
                         'hardship_last_payment_amount', 'disbursement_method', 'debt_settlement_flag', 'debt_settlement_flag_date', 
                         'settlement_status', 'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term']

In [6]:
dataset = dataset.drop(columns=features_to_eliminate_lo)

In [7]:
dataset.shape

(226067, 109)

#### 1.2 Removing loans with Joint App application type

In [8]:
dataset['application_type'].value_counts(normalize=True) * 100

application_type
Individual    94.689185
Joint App      5.310815
Name: proportion, dtype: float64

In [9]:
dataset = dataset[dataset['application_type']=='Individual']

In [10]:
dataset['application_type'].value_counts(normalize=True) * 100

application_type
Individual    100.0
Name: proportion, dtype: float64

In [11]:
dataset.shape

(214061, 109)

##### 1.2.1 Removing loan_status and the features exclusive to Joint App loans

In [12]:
features_to_eliminate_ja = ['loan_status', 'application_type','annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq']

In [13]:
dataset = dataset.drop(columns=features_to_eliminate_ja)

In [14]:
dataset.shape

(214061, 103)

#### 1.2 Identifying prevalence of missing data per field

In [15]:
def top_missing_cols_perc(df):
    """
    returns missing columns in dataframe with missing values percent > thresh
    if n=None. It will gave whole dataframe with missing values percent > thresh
    """
    
    dff = (df.isnull().sum()/df.shape[0])*100
    dff = dff.reset_index()
    dff.columns = ['col','missing_percent']
    dff = dff.sort_values(by=['missing_percent'],ascending=True).reset_index(drop=True)
    return dff

In [16]:
missing_perc_matrix = top_missing_cols_perc(dataset)

In [17]:
missing_perc_matrix.groupby('missing_percent').count().reset_index().to_csv("missing_perc_matrix.csv", index=False)

In [18]:
missing_perc_matrix.to_csv("missing_perc_matrix_values.csv", index=False)

plt.figure(figsize=(10, 4))
plt.hist(missing_perc_matrix.groupby('missing_percent').count().reset_index()['missing_percent'],bins=20,color = "skyblue", ec="#8FAADC")
plt.savefig('graphs/missing_data_histogram',dpi=300)

In [19]:
missing_perc_matrix.to_csv("intermediate_output/missing_perc_matrix_values.csv", index=False)

In [20]:
dataset.head(100).to_csv('intermediate_output/check_final_data.csv',index=False)

#### 1.2 Remove features with 100% missing values

In [21]:
missing_perc_matrix100 = missing_perc_matrix[missing_perc_matrix['missing_percent']==100]

In [22]:
missing_perc_matrix100.shape

(14, 2)

In [23]:
features_missing100 = missing_perc_matrix100['col'].to_list()

In [24]:
dataset = dataset.drop(columns=features_missing100)

In [25]:
dataset.shape

(214061, 89)

#### 1.3 Correlation Analysis

##### 1.3.1 Get All possible Correlations with features

In [26]:
all_features = dataset.columns.to_list()

In [27]:
len(all_features)

89

In [28]:
string_columns_all = ['term', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'issue_d',
                      'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'earliest_cr_line', 'initial_list_status']
#string_columns_all = dataset.select_dtypes(include=['object']).columns.tolist()
string_columns_all

['term',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'verification_status',
 'issue_d',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'earliest_cr_line',
 'initial_list_status']

In [29]:
le = LabelEncoder()
for column in string_columns_all:
    dataset[column] = le.fit_transform(dataset[column])

In [30]:
corr_matrix = dataset.corr()

In [31]:
mask = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
item_pairs = corr_matrix.where(mask)

In [32]:
correlation_df = item_pairs.stack().reset_index()
correlation_df.columns = ['Feature 1', 'Feature 2', 'Correlation']

In [33]:
correlation_df.head()

,Feature 1,Feature 2,Correlation
0,loan_amnt,funded_amnt,0.999734
1,loan_amnt,funded_amnt_inv,0.998951
2,loan_amnt,term,0.391559
3,loan_amnt,int_rate,0.099515
4,loan_amnt,grade,0.104163


In [34]:
#correlation_df['Correlation'] = correlation_df['Correlation'].apply(lambda x: x*100)
correlation_df['Abs_Correlation'] = correlation_df['Correlation'].apply(lambda x: abs(x))

##### 1.3.2 Separate default_status related correlation

In [35]:
def feature_(cols):
    feature1 = cols['Feature 1']
    feature2 = cols['Feature 2']
    if feature1=='default_status':
        return feature2
    else:
        return feature1

In [36]:
label_corr = correlation_df[(correlation_df['Feature 1']=='default_status')|(correlation_df['Feature 2']=='default_status')].copy()
label_corr['Feature'] = label_corr.apply(feature_,axis=1)

In [37]:
label_corr = label_corr.drop(columns=['Feature 1', 'Feature 2'])

In [38]:
label_corr.rename(columns={'Correlation':'label_corr', 'Abs_Correlation':'label_abs_corr'},inplace=True)
label_corr

,label_corr,label_abs_corr,Feature
86,0.026821,0.026821,loan_amnt
172,0.026788,0.026788,funded_amnt
257,0.026352,0.026352,funded_amnt_inv
341,0.091114,0.091114,term
424,0.198494,0.198494,int_rate
...,...,...,...
3817,-0.061955,0.061955,tot_hi_cred_lim
3821,-0.010338,0.010338,total_bal_ex_mort
3824,-0.072733,0.072733,total_bc_limit
3826,-0.012104,0.012104,total_il_high_credit_limit


##### 1.2.3 Get a Feature only table and add correlation to default_status on the feature tables

In [39]:
features_corr = correlation_df[(correlation_df['Feature 1']!='default_status')&(correlation_df['Feature 2']!='default_status')]
features_corr = features_corr.merge(label_corr[['Feature','label_abs_corr']],left_on=['Feature 1'], right_on=['Feature'], how='left')

In [40]:
features_corr.rename(columns={'label_abs_corr':'label_abs_corr_feat1'},inplace=True)
features_corr = features_corr.drop(columns=['Feature'])

In [41]:
features_corr = features_corr.merge(label_corr[['Feature','label_abs_corr']],left_on=['Feature 2'], right_on=['Feature'], how='left')

In [42]:
features_corr.rename(columns={'label_abs_corr':'label_abs_corr_feat2'},inplace=True)
features_corr = features_corr.drop(columns=['Feature'])

##### 1.3.4 Add Missing Prevalences to features dataset

In [43]:
features_corr = features_corr.merge(missing_perc_matrix,left_on=['Feature 1'], right_on=['col'], how='left')
features_corr.rename(columns={'missing_percent':'Feature 1 Missing'},inplace=True)
features_corr = features_corr.drop(columns=['col'])

In [44]:
features_corr = features_corr.merge(missing_perc_matrix,left_on=['Feature 2'], right_on=['col'], how='left')
features_corr.rename(columns={'missing_percent':'Feature 2 Missing'},inplace=True)
features_corr = features_corr.drop(columns=['col'])

In [45]:
features_corr

,Feature 1,Feature 2,Correlation,Abs_Correlation,label_abs_corr_feat1,label_abs_corr_feat2,Feature 1 Missing,Feature 2 Missing
0,loan_amnt,funded_amnt,0.999734,0.999734,0.026821,0.026788,0.000000,0.00000
1,loan_amnt,funded_amnt_inv,0.998951,0.998951,0.026821,0.026352,0.000000,0.00000
2,loan_amnt,term,0.391559,0.391559,0.026821,0.091114,0.000000,0.00000
3,loan_amnt,int_rate,0.099515,0.099515,0.026821,0.198494,0.000000,0.00000
4,loan_amnt,grade,0.104163,0.104163,0.026821,0.221883,0.000000,0.00000
...,...,...,...,...,...,...,...,...
3736,total_bal_ex_mort,total_il_high_credit_limit,0.874240,0.874240,0.010338,0.012104,2.360542,3.32522
3737,total_bal_ex_mort,year_issued,0.036033,0.036033,0.010338,0.146005,2.360542,0.00000
3738,total_bc_limit,total_il_high_credit_limit,0.139136,0.139136,0.072733,0.012104,2.360542,3.32522
3739,total_bc_limit,year_issued,0.097837,0.097837,0.072733,0.146005,2.360542,0.00000


In [46]:
features_corr.to_csv('intermediate_output/features_corr.csv',index=False)

#### Result 1: Pick Combinations with correlations higher than 0.9

In [47]:
features_corr_v90 = features_corr[features_corr['Abs_Correlation']>0.9].copy()

In [48]:
def feature_to_go(df):
    feat1_missing = df['Feature 1 Missing']
    feat2_missing = df['Feature 2 Missing']
    feat1 = df['Feature 1']
    feat2 = df['Feature 2']
    feat1_label_corr = df['label_abs_corr_feat1']
    feat2_label_corr = df['label_abs_corr_feat2']
    if feat1_missing==feat2_missing:
        if feat1_label_corr<feat2_label_corr:
            return feat1
        else:
            return feat2
    else:
        if feat1_missing<feat2_missing:
            return feat2
        else:
            return feat1

In [49]:
features_corr_v90['Feature which goes'] = features_corr_v90.apply(feature_to_go, axis=1)

In [50]:
intermediate_output_path = os.path.join(r'intermediate_output', 'features_corr_v90.csv')
features_corr_v90.to_csv(intermediate_output_path, index=False)

In [51]:
features_to_go = features_corr_v90['Feature which goes'].tolist()

In [52]:
features_to_go = list(dict.fromkeys(features_to_go))

In [53]:
len(features_to_go)

8

In [54]:
len(features_to_go)

8

In [55]:
pd.DataFrame(features_to_go).to_csv('intermediate_output/features_to_go.csv',index=False)

##### Analysing the remaining features

In [56]:
all_features = dataset.columns.tolist()

In [57]:
features_to_stay = [x for x in all_features if x not in features_to_go]

In [58]:
len(features_to_stay)

81

#### Result 2: For the remaining features, analyse correlation coefficient with respect to the label for remaining features with >80% missing value

In [59]:
features_to_stay_df = pd.DataFrame(features_to_stay, columns=['Features to stay'])

In [60]:
features_to_stay_df.head(1)

,Features to stay
0,loan_amnt


In [61]:
features_to_stay_df.to_csv('features_to_stay.csv', index=False)

In [62]:
### Get label_corr with missing values
label_corr = label_corr.copy()
label_corr = label_corr.merge(missing_perc_matrix,left_on=['Feature'], right_on=['col'], how='left')
label_corr = label_corr.drop(columns=['col'])

In [63]:
label_corr_2 = label_corr.merge(features_to_stay_df,left_on=['Feature'], right_on=['Features to stay'], how='left')

In [64]:
### Remove features which have already been taken out in step 1
label_corr_2 = label_corr_2[label_corr_2['Features to stay'].notna()]

In [65]:
label_corr_2

,label_corr,label_abs_corr,Feature,missing_percent,Features to stay
0,0.026821,0.026821,loan_amnt,0.000000,loan_amnt
3,0.091114,0.091114,term,0.000000,term
6,0.225005,0.225005,sub_grade,0.000000,sub_grade
7,0.025227,0.025227,emp_title,6.959231,emp_title
8,0.022128,0.022128,emp_length,6.086583,emp_length
...,...,...,...,...,...
82,-0.061955,0.061955,tot_hi_cred_lim,3.325220,tot_hi_cred_lim
83,-0.010338,0.010338,total_bal_ex_mort,2.360542,total_bal_ex_mort
84,-0.072733,0.072733,total_bc_limit,2.360542,total_bc_limit
85,-0.012104,0.012104,total_il_high_credit_limit,3.325220,total_il_high_credit_limit


In [66]:
label_corr_2.to_csv('intermediate_output/label_corr.csv', index=False)

##### Analyse the features sets - refer to the thesis report for methodology

In [67]:
def get_feature_sets(df):
    initial_features = df['Feature'].tolist()
    features_to_go = []
    features_to_stay = []
    features_checked = 0

    feature_subset = pd.DataFrame() 
    while features_checked < len(initial_features):
        df.sort_values(by='label_abs_corr', ascending=True, inplace=True)
        df['Cumulative ABS Corr'] = df['label_abs_corr'].cumsum()
        feature_subset = df[df['Cumulative ABS Corr']<=0.3]
        if feature_subset.shape[0]!=0:
            for feature in feature_subset['Feature'].tolist():
                features_to_go.append(feature)
                features_checked = len(features_to_go)
                
        df  = df[df['Cumulative ABS Corr']>0.3].copy()

    features_to_stay = [x for x in initial_features if x not in features_to_go]
    return features_to_go, features_to_stay

In [68]:
label_corr_U40 = label_corr_2[label_corr_2['missing_percent']>40].copy()

In [69]:
features_to_go2, features_to_stay2 = get_feature_sets(label_corr_U40)

In [70]:
features_to_go2

['mths_since_recent_bc_dlq',
 'mths_since_recent_revol_delinq',
 'open_act_il',
 'total_cu_tl',
 'mths_since_last_major_derog',
 'mths_since_last_delinq',
 'desc',
 'mths_since_last_record',
 'max_bal_bc',
 'mths_since_rcnt_il',
 'inq_fi',
 'open_rv_12m',
 'il_util',
 'open_il_12m',
 'open_il_24m',
 'open_acc_6m',
 'inq_last_12m',
 'open_rv_24m',
 'all_util']

In [71]:
features_to_stay2

[]

#### Dataset with the remaining set of features after step 1

In [72]:
#features_to_stay

In [73]:
final_features_to_stay = [x for x in features_to_stay if x not in features_to_go2]

In [74]:
len(final_features_to_stay)

62

In [75]:
intermediate_output_path = os.path.join(r'intermediate_output', 'final_features_to_stay.csv')
pd.DataFrame(final_features_to_stay, columns=["features_to_stay"]).to_csv(intermediate_output_path, index=False)